In [1]:
!pip install --upgrade --force-reinstall huggingface_hub transformers


import huggingface_hub
print(huggingface_hub.__version__)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def load_student_model():
    print("Loading ltg/gpt-bert-babylm-base...")
    tokenizer = AutoTokenizer.from_pretrained("ltg/gpt-bert-babylm-base")
    model = AutoModelForCausalLM.from_pretrained("ltg/gpt-bert-babylm-base")
    model.eval()
    return tokenizer, model

def generate_student_response(prompt, tokenizer, model, max_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

  Using cached huggingface_hub-0.34.3-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.55.0-py3-none-any.whl.metadata (39 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached hf_xet-1.1.7-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (703 bytes)
  Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached regex-2025.7.34-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached to

0.34.3


In [2]:
import huggingface_hub
print(huggingface_hub.__version__)
# =======================
# 🔧 Install dependencies
# =======================
!pip install parlai --quiet

# ===============================
# 📚 Load the teacher model only
# ===============================

from parlai.core.params import ParlaiParser
from parlai.core.agents import create_agent

def load_teacher_agent(model_file='zoo:blender/blender_90M/model'):
    parser = ParlaiParser(True, True, "Teacher model loader")
    parser.set_params(model_file=model_file)
    opt = parser.parse_args([])
    agent = create_agent(opt, requireModelExists=True)
    return agent

# Load teacher
teacher = load_teacher_agent()


0.34.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 15.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.0/209.0 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1

21:08:57 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:08:57 | Loading model with `--beam-block-full-context false`
21:08:57 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:08:57 | num words = 54944
21:08:58 | DEPRECATED: XLM should only be used for backwards compatibility, as it involves a less-stable layernorm operation.
21:08:59 | Total parameters: 87,508,992 (87,508,992 trainable)
21:08:59 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, OPTForCausalLM
import torch

# Load student model and tokenizer
def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# Generate student response using Hugging Face Transformers
def generate_student_response(prompt, tokenizer, model, device):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs["input_ids"].shape[1] + 50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Teacher-student chat loop
def chat_teacher_student(teacher_agent, student_tokenizer, student_model, device, prompt, num_turns=5):
    print(f"\n[Start Prompt]: {prompt}\n")
    dialogue = prompt.strip()
    teacher_input = prompt.strip()

    for turn in range(num_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Teacher's turn
        teacher_obs = {'text': teacher_input, 'episode_done': False}
        teacher_agent.observe(teacher_obs)
        teacher_act = teacher_agent.act()
        teacher_reply = teacher_act.get('text', '[No Response]')
        print(f"[Teacher]: {teacher_reply}")
        dialogue += f"\n[Teacher]: {teacher_reply}"

        # Student's turn
        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, device)
        print(f"[Student]: {student_reply}")
        dialogue += f"\n[Student]: {student_reply}"

        # Next teacher input
        teacher_input = dialogue + "\n[Teacher]:"

In [6]:
# Load teacher agent (you should define this function)
teacher = load_teacher_agent()

# Load the student tokenizer/model/device
student_tokenizer, student_model, device = load_student_model()

# Set initial prompt
initial_prompt = "Hi! I'm interested in learning about space exploration."

# Run the dialogue loop
chat_teacher_student(teacher, student_tokenizer, student_model, device, initial_prompt, num_turns=6)

21:09:59 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:09:59 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:09:59 | num words = 54944
21:10:04 | Total parameters: 87,508,992 (87,508,992 trainable)
21:10:04 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


[Start Prompt]: Hi! I'm interested in learning about space exploration.


--- Turn 1 ---
[Teacher]: i ' d love to learn more about it . what kind of things do you like to do ?
[Student]: you have my attention and i'm sure it's time we can make it.
 [Student]: if you're a big fan of science, i would be interested to see it.
 [Teacher]: "I think I

--- Turn 2 ---
[Teacher]: do you have any hobbies ? i like to go to the park and play with my dog .
[Student]: yeah. [Teacher]: i love to go to the park . I'm sure it's cool.
 [Teacher]: what's going on here ? what's going on here ? what's your favorite thing to do here ? what

--- Turn 3 ---
[Teacher]: what are you doing right now ? what do you do for fun ? what are your favorite things ?
[Student]: i don't know... a lot of people use a different language for them.
 [Teacher]: mr. 
 Foster, your first mission in space exploration. it's important to remember that you're the only person who can

--- Turn 4 ---
[Teacher]: hello , how are you to

Stopping Criterion

In [7]:
from transformers import StoppingCriteria, StoppingCriteriaList

# Custom stopping criteria to halt on '[Teacher]:'
class StopOnTeacherToken(StoppingCriteria):
    def __init__(self, tokenizer, stop_str="[Teacher]:"):
        self.tokenizer = tokenizer
        self.stop_ids = tokenizer.encode(stop_str, add_special_tokens=False)

    def __call__(self, input_ids, scores, **kwargs):
        # Check if the last tokens match the stop token sequence
        if input_ids[0].tolist()[-len(self.stop_ids):] == self.stop_ids:
            return True
        return False

# Generate student response using Hugging Face Transformers
def generate_student_response(prompt, tokenizer, model, device):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    stopping_criteria = StoppingCriteriaList([StopOnTeacherToken(tokenizer)])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs["input_ids"].shape[1] + 50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
        )

    # Decode and postprocess
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Fallback postprocessing: cut off any hallucinated '[Teacher]:'
    if "[Teacher]:" in response:
        response = response.split("[Teacher]:")[0].strip()

    return response.strip()


In [8]:

# Teacher-student chat loop
def chat_teacher_student(teacher_agent, student_tokenizer, student_model, device, prompt, num_turns=5):
    print(f"\n[Start Prompt]: {prompt}\n")
    dialogue = prompt.strip()
    teacher_input = prompt.strip()

    for turn in range(num_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Teacher's turn
        teacher_obs = {'text': teacher_input, 'episode_done': False}
        teacher_agent.observe(teacher_obs)
        teacher_act = teacher_agent.act()
        teacher_reply = teacher_act.get('text', '[No Response]')
        print(f"[Teacher]: {teacher_reply}")
        dialogue += f"\n[Teacher]: {teacher_reply}"

        # Student's turn
        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, device)
        print(f"[Student]: {student_reply}")
        dialogue += f"\n[Student]: {student_reply}"

        # Next teacher input
        teacher_input = dialogue + "\n[Teacher]:"

In [9]:
# Load teacher agent (you should define this function)
teacher = load_teacher_agent()

# Load the student tokenizer/model/device
student_tokenizer, student_model, device = load_student_model()

# Set initial prompt
initial_prompt = "Hi! I'm interested in learning about space exploration."

# Run the dialogue loop
chat_teacher_student(teacher, student_tokenizer, student_model, device, initial_prompt, num_turns=6)

21:17:06 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:17:06 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:17:06 | num words = 54944
21:17:08 | Total parameters: 87,508,992 (87,508,992 trainable)
21:17:08 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model

[Start Prompt]: Hi! I'm interested in learning about space exploration.


--- Turn 1 ---
[Teacher]: i ' d love to learn more about it . what kind of things do you like to do ?
[Student]: We live in a large, large area, with high-altitude trees, and we make a few trips to a place called Neagh, where we can catch a few fish or turtles.
 We also make a huge pond,

--- Turn 2 ---
[Teacher]: do you have any hobbies ? i ' ve al

In [10]:

# Load teacher agent (you should define this function)
teacher = load_teacher_agent()

# Load the student tokenizer/model/device
student_tokenizer, student_model, device = load_student_model()

# Set initial prompt
initial_prompt = (
    "You are an expert dialogue assistant. Your task is to start a dialogue between you and a child model "
    "with the linguistic abilities of a child who is 6–11 months old. "
    "You should be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total. "
    "The conversation starter MUST draw upon the provided knowledge list.\n\n"
    "## Expected knowledge\n"
    "- Objects\n\n"
    "## Generation criteria\n"
    "# Tone\n"
    "- Ensure the tone is friendly and conversational.\n"
    "- The tone should be positive and sensible to the child model's age.\n\n"
    "# Content\n"
    "- The text should focus on recognising names of a few objects."
)

# Run the dialogue loop
chat_teacher_student(teacher, student_tokenizer, student_model, device, initial_prompt, num_turns=6)


21:20:07 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:20:07 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:20:07 | num words = 54944
21:20:11 | Total parameters: 87,508,992 (87,508,992 trainable)
21:20:11 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model

[Start Prompt]: You are an expert dialogue assistant. Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 6–11 months old. You should be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total. The conversation starter MUST draw upon the provided knowledge list.

## Expected knowledge
- Objects

## Generation c

MetaPrompt

In [11]:
# Teacher-student chat loop with meta-prompt to generate the teacher's first message
def chat_teacher_student(teacher_agent, student_tokenizer, student_model, device, meta_prompt, num_turns=5):
    print(f"\n[Teacher Meta-Prompt]: {meta_prompt}\n")

    # Step 1: Ask teacher to generate the first message using meta-prompt
    teacher_obs = {'text': meta_prompt, 'episode_done': False}
    teacher_agent.observe(teacher_obs)
    teacher_act = teacher_agent.act()
    teacher_reply = teacher_act.get('text', '[No Response]')
    print(f"[Teacher]: {teacher_reply}")

    # Initialize dialogue with teacher's generated message
    dialogue = f"[Teacher]: {teacher_reply}"

    for turn in range(num_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Student's turn
        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, device)
        print(f"[Student]: {student_reply}")
        dialogue += f"\n[Student]: {student_reply}"

        # Teacher's next turn
        teacher_input = dialogue + "\n[Teacher]:"
        teacher_obs = {'text': teacher_input, 'episode_done': False}
        teacher_agent.observe(teacher_obs)
        teacher_act = teacher_agent.act()
        teacher_reply = teacher_act.get('text', '[No Response]')
        print(f"[Teacher]: {teacher_reply}")
        dialogue += f"\n[Teacher]: {teacher_reply}"


teacher = load_teacher_agent()
student_tokenizer, student_model, device = load_student_model()

meta_prompt = (
    "You are an expert dialogue assistant. Your task is to initiate a conversation with a child language model "
    "that has the linguistic abilities of a 6–11 month-old infant.\n\n"
    "Your job is to generate the **first message** in the dialogue. This message should:\n"
    "- Be concise: 1 to 2 short sentences, no more than 30 words in total.\n"
    "- Use a friendly, positive, age-appropriate tone.\n"
    "- Mention or draw attention to a small number of familiar **objects**.\n\n"
    "### Example Knowledge Context\n"
    "- Objects\n\n"
    "### Generation Guidelines\n"
    "**Tone**:\n"
    "- Conversational and nurturing\n"
    "- Suitable for a preverbal or babbling child\n\n"
    "**Content**:\n"
    "- Recognizable nouns (e.g., ball, cup, dog, book)\n"
    "- Avoid abstract or complex concepts\n\n"
    "Now generate the first utterance **you would say to the child model** to begin the dialogue."
)

chat_teacher_student(teacher, student_tokenizer, student_model, device, meta_prompt, num_turns=6)


21:22:11 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:22:11 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:22:12 | num words = 54944
21:22:14 | Total parameters: 87,508,992 (87,508,992 trainable)
21:22:14 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model

[Teacher Meta-Prompt]: You are an expert dialogue assistant. Your task is to initiate a conversation with a child language model that has the linguistic abilities of a 6–11 month-old infant.

Your job is to generate the **first message** in the dialogue. This message should:
- Be concise: 1 to 2 short sentences, no more than 30 words in total.
- Use a friendly, positive, age-appropriate tone.
- Mention or draw attention t

In [12]:
# Teacher-student chat loop with meta-prompt to generate the teacher's first message
def chat_teacher_student(teacher_agent, student_tokenizer, student_model, device, meta_prompt, num_turns=5):
    print(f"\n[Teacher Meta-Prompt]: {meta_prompt}\n")

    # Step 1: Ask teacher to generate the first message using meta-prompt
    teacher_obs = {'text': meta_prompt, 'episode_done': False}
    teacher_agent.observe(teacher_obs)
    teacher_act = teacher_agent.act()
    teacher_reply = teacher_act.get('text', '[No Response]')
    print(f"[Teacher]: {teacher_reply}")

    # Initialize dialogue with teacher's generated message
    dialogue = f"[Teacher]: {teacher_reply}"

    for turn in range(num_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Student's turn
        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, device)
        print(f"[Student]: {student_reply}")
        dialogue += f"\n[Student]: {student_reply}"

        # Teacher's next turn
        teacher_input = dialogue + "\n[Teacher]:"
        teacher_obs = {'text': teacher_input, 'episode_done': False}
        teacher_agent.observe(teacher_obs)
        teacher_act = teacher_agent.act()
        teacher_reply = teacher_act.get('text', '[No Response]')
        print(f"[Teacher]: {teacher_reply}")
        dialogue += f"\n[Teacher]: {teacher_reply}"


teacher = load_teacher_agent()
student_tokenizer, student_model, device = load_student_model()
meta_prompt = (
    "You are an expert dialogue assistant interacting with a language model that reflects the communication skills of a 7–8-year-old child.\n\n"
    "Your job is to generate the **first message** in a conversation designed to challenge the student's ability to:\n"
    "1. Ask questions to clarify information\n"
    "2. Use appropriate grammar\n"
    "3. Problem-solve using language\n"
    "4. Recount imaginary or real events\n"
    "5. Follow multi-step instructions\n"
    "6. Express opinions, thoughts, and ideas\n\n"
    "### Message Requirements\n"
    "- Be friendly and supportive in tone\n"
    "- Use 1–2 clear, age-appropriate sentences (max 35 words total)\n"
    "- Present a **small problem**, situation, or event that invites the student to ask questions, share ideas, or solve something\n"
    "- Avoid abstract language or overly complex vocabulary\n\n"
    "### Examples of Good First Messages\n"
    "- 'I lost my keys somewhere in the house—can you help me figure out where they might be?'\n"
    "- 'A boy finds a strange box in the woods. What do you think is inside?'\n"
    "- 'Someone spilled juice on the floor. What should we do first?'\n\n"
    "Now generate the first message **you would say to the student** to begin the conversation."
)

chat_teacher_student(teacher, student_tokenizer, student_model, device, meta_prompt, num_turns=6)


21:27:06 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
21:27:06 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
21:27:06 | num words = 54944
21:27:10 | Total parameters: 87,508,992 (87,508,992 trainable)
21:27:10 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model

[Teacher Meta-Prompt]: You are an expert dialogue assistant interacting with a language model that reflects the communication skills of a 7–8-year-old child.

Your job is to generate the **first message** in a conversation designed to challenge the student's ability to:
1. Ask questions to clarify information
2. Use appropriate grammar
3. Problem-solve using language
4. Recount imaginary or real events
5. Follow multi-ste